# Clasificación automática de residuos mediante visión artificial
## Sistemas de percepción en robótica · Implementación técnica

**Objetivo:** clasificar una fotografía con un único residuo principal como
`carton`, `metal`, `plastico` o `vidrio`, mediante ResNet-18 y transfer learning.
La clase y la confianza alimentan una decisión robótica **simulada**.

**Flujo:** imagen → preprocesamiento → ResNet-18 → clase y confianza →
contenedor correspondiente o revisión manual.

Este notebook contiene la parte técnica: entorno, preparación del pipeline,
entrenamiento, validación, test, predicción e integración de la decisión.
La selección, recopilación y preparación del dataset corresponden al resto
del grupo. No se incluyen imágenes artificiales, resultados inventados,
memoria final, presentación ni conclusiones experimentales.

**Estado de entrega:** código preparado; ejecución comprobada hasta la
validación del dataset. Las etapas posteriores requieren las imágenes
definitivas y un entrenamiento real.

## 1. Preparación del entorno

**Las dependencias se instalan una sola vez desde requirements.txt.**

Selecciona el kernel **Python - Clasificador Residuos**. Este notebook no
ejecuta `pip install` durante `Run All`. Si cambia el entorno, usa desde una
terminal el ejecutable que aparece en la siguiente celda y la opción
`-m pip install -r requirements.txt`; no uses un `pip` de otro Python.

Abre este notebook desde su carpeta de proyecto, junto a `requirements.txt`.
Ejecuta en orden. Si faltan imágenes, el flujo se detendrá en el apartado 4
con un mensaje breve. No se descargan datasets ni pesos del modelo en esa fase.

## Comprobación del entorno

In [ ]:
import sys
import os
import platform
import importlib.metadata as metadata
import torch
import torchvision
import numpy as np
import sklearn
import matplotlib
from PIL import Image

## 2. Importaciones

`ImageFolder` obtiene las etiquetas de las carpetas. Scikit-learn realiza la
división estratificada y calcula las métricas. Pillow abre imágenes y
Matplotlib representa ejemplos, curvas y la matriz de confusión.

In [ ]:
import copy
import csv
import json
import math
import random
import time
from datetime import datetime
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torchvision.datasets.folder import IMG_EXTENSIONS
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)
from IPython.display import display, Markdown

## 3. Configuración reproducible y dispositivo

Se fija una semilla para reproducir la división y reducir variaciones del
entrenamiento. La reproducibilidad exacta entre máquinas o versiones no está
garantizada. Se usa CUDA solo cuando PyTorch la detecta; en otro caso, CPU.
`NUM_WORKERS = 0` facilita el funcionamiento de los DataLoaders en Windows/Jupyter.

Los porcentajes 70/15/15 son una configuración inicial modificable del
experimento; se utilizan todas las imágenes que entregue el grupo, sin
seleccionar una cantidad por clase. El test se reserva para la evaluación
final y no se usa para elegir hiperparámetros ni el umbral de decisión.

`ENTRENAR = False` evita iniciar un entrenamiento accidental. Actívalo cuando
el dataset esté entregado y validado. La primera carga de pesos preentrenados
puede necesitar conexión a Internet; esto solo ocurre al activar el entrenamiento.

In [ ]:
SEED = 42
CLASES = ["carton", "metal", "plastico", "vidrio"]
DATASET_PATH = Path("dataset")
PRUEBAS_PATH = Path("pruebas")
MODELOS_PATH = Path("modelos")
RESULTADOS_PATH = Path("resultados")

FRACCION_TRAIN, FRACCION_VAL, FRACCION_TEST = 0.70, 0.15, 0.15
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS_CABEZA = 15
EPOCHS_FINE_TUNING = 8
PACIENCIA = 4
MIN_DELTA = 1e-4
LR_CABEZA = 1e-3
LR_FINE_TUNING = 1e-5
WEIGHT_DECAY = 1e-4
UMBRAL_CONFIANZA = 0.70  # Valor inicial, pendiente de ajustar con validación.
RATIO_DESEQUILIBRIO = 3.0  # Advertencia técnica; no modifica las imágenes.
ENTRENAR = False
HACER_FINE_TUNING = False
EVALUAR_TEST = False
CHECKPOINT_A_CARGAR = None  # Ejemplo de ruta relativa: Path("modelos/resnet18_....pt")
IMAGEN_A_PREDECIR = None   # Ejemplo de ruta relativa: Path("pruebas/imagen.jpg")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_OK = False
MODELO_ENTRENADO = False
modelo = None
historial = []
resultado_prediccion = None
resultados_test = None
print("Dispositivo:", DEVICE)
print("Entrenamiento activado:", ENTRENAR)

### Estructura esperada

```text
Clasificador_Residuos.ipynb
requirements.txt
dataset/
    carton/
    metal/
    plastico/
    vidrio/
pruebas/
modelos/
resultados/
```

La siguiente celda solo crea las carpetas vacías que falten. No mueve,
renombra, borra, etiqueta ni descarga imágenes.

In [ ]:
for carpeta in [DATASET_PATH, PRUEBAS_PATH, MODELOS_PATH, RESULTADOS_PATH]:
    carpeta.mkdir(parents=True, exist_ok=True)
for clase in CLASES:
    (DATASET_PATH / clase).mkdir(exist_ok=True)
print("Estructura de carpetas preparada.")

## 4. Comprobación del dataset · punto de espera

Se comprueban carpetas, extensiones y número de archivos por clase. Las
extensiones no admitidas se notifican y no se usan. Una clase vacía o una
estructura distinta de las cuatro clases impide continuar. Un desequilibrio
superior al umbral configurado genera una advertencia, sin alterar los datos.

Esta comprobación no sustituye la revisión y el etiquetado del dataset por
los compañeros. No elimina duplicados ni verifica manualmente el contenido.
El grupo debe mantener imágenes del mismo objeto o sesión de captura en
una sola partición para evitar fuga de información; la división automática
por imágenes no puede reconocer esa relación sin metadatos adicionales.

**Si todavía no hay imágenes, es correcto que la ejecución termine aquí.**
El mensaje `DatasetPendiente` es una parada controlada, sin un traceback extenso.

In [ ]:
class DatasetPendiente(Exception):
    def _render_traceback_(self):
        return [str(self)]

def comprobar_dataset(ruta):
    ruta = Path(ruta)
    problemas, conteos = [], {}
    if not ruta.is_dir():
        return False, {}, [f"No existe la carpeta {ruta}."]
    encontradas = {p.name for p in ruta.iterdir() if p.is_dir()}
    if encontradas != set(CLASES):
        problemas.append(f"Carpetas esperadas: {CLASES}; encontradas: {sorted(encontradas)}.")
    for clase in CLASES:
        carpeta = ruta / clase
        archivos = [p for p in carpeta.rglob("*") if p.is_file()] if carpeta.is_dir() else []
        validas = [p for p in archivos if p.suffix.lower() in IMG_EXTENSIONS]
        conteos[clase] = len(validas)
        print(f"{clase}: {len(validas)} imágenes con extensión admitida")
        ignorados = len(archivos) - len(validas)
        if ignorados:
            print(f"  Aviso: {ignorados} archivos con extensión no admitida; no se modifican.")
        if not validas:
            problemas.append(f"La clase {clase} está vacía o no contiene extensiones admitidas.")
    if conteos and min(conteos.values()) > 0:
        ratio = max(conteos.values()) / min(conteos.values())
        if ratio > RATIO_DESEQUILIBRIO:
            print(f"Aviso: desequilibrio máximo/mínimo = {ratio:.2f}. Comunicar al grupo.")
    return not problemas, conteos, problemas

DATASET_OK, conteos, problemas = comprobar_dataset(DATASET_PATH)
if not DATASET_OK:
    display(Markdown("**Esperando el dataset del grupo.** No se ejecutan entrenamiento ni evaluación."))
    raise DatasetPendiente("\n".join(problemas) + "\nAñade el dataset entregado por el grupo y vuelve a ejecutar desde el principio.")
print("Estructura válida. La viabilidad de la división se comprueba en el apartado 6.")

## 5. Transformaciones

El entrenamiento incorpora variaciones moderadas de recorte, giro, inversión
horizontal y color. Validación, test e inferencia usan un preprocesamiento
determinista, coherente con los pesos de ResNet-18: tamaño de entrada 224 × 224
y normalización de ImageNet. Las transformaciones se aplican al cargar los
lotes y no generan ni guardan imágenes para sustituir el dataset.

In [ ]:
PESOS = ResNet18_Weights.IMAGENET1K_V1
transform_eval = PESOS.transforms()  # No descarga los pesos.
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.80, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

## 6. ImageFolder y división estratificada

Se crean dos vistas del mismo conjunto: una con aumento de datos para train
y otra determinista para validación y test. Los índices se dividen una sola
vez y se verifica que sean disjuntos y que las cuatro clases estén presentes
en cada partición. Si los datos no permiten estratificar, se comunica el
problema al grupo; el código no añade ni elimina imágenes para resolverlo.

In [ ]:
if not DATASET_OK:
    raise DatasetPendiente("Revisa primero la comprobación del dataset.")
dataset_train = datasets.ImageFolder(DATASET_PATH, transform=transform_train)
dataset_eval = datasets.ImageFolder(DATASET_PATH, transform=transform_eval)
if dataset_train.classes != CLASES or dataset_eval.samples != dataset_train.samples:
    raise ValueError("Las clases o el orden de las dos vistas no coinciden.")
if not math.isclose(FRACCION_TRAIN + FRACCION_VAL + FRACCION_TEST, 1.0):
    raise ValueError("Las fracciones deben sumar 1.")
if min(FRACCION_TRAIN, FRACCION_VAL, FRACCION_TEST) <= 0:
    raise ValueError("Las tres fracciones deben ser positivas.")

etiquetas = np.array(dataset_eval.targets)
indices = np.arange(len(etiquetas))
try:
    idx_train, idx_resto = train_test_split(
        indices, test_size=FRACCION_VAL + FRACCION_TEST,
        stratify=etiquetas, random_state=SEED,
    )
    idx_val, idx_test = train_test_split(
        idx_resto, test_size=FRACCION_TEST / (FRACCION_VAL + FRACCION_TEST),
        stratify=etiquetas[idx_resto], random_state=SEED,
    )
except ValueError as error:
    raise DatasetPendiente(f"El dataset no permite esta división estratificada: {error}") from None

splits = {"train": idx_train, "val": idx_val, "test": idx_test}
for nombre, ids in splits.items():
    if set(etiquetas[ids]) != set(range(len(CLASES))):
        raise DatasetPendiente(f"Faltan clases en {nombre}; revisar la división con el grupo.")
    print(nombre, len(ids), {CLASES[k]: int(v) for k, v in sorted(Counter(etiquetas[ids]).items())})
assert not set(idx_train) & set(idx_val)
assert not set(idx_train) & set(idx_test)
assert not set(idx_val) & set(idx_test)
assert len(set(idx_train) | set(idx_val) | set(idx_test)) == len(indices)

particiones = [
    {"archivo": Path(dataset_eval.samples[int(i)][0]).relative_to(DATASET_PATH).as_posix(),
     "clase": CLASES[int(etiquetas[i])], "particion": nombre}
    for nombre, ids in splits.items() for i in ids
]

## 7. DataLoaders

Solo se baraja el entrenamiento. El orden de test se conserva para relacionar
cada predicción con su imagen. No se comparte la transformación aleatoria
de train con las otras particiones.

In [ ]:
generador = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(Subset(dataset_train, idx_train.tolist()), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, generator=generador)
val_loader = DataLoader(Subset(dataset_eval, idx_val.tolist()), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(Subset(dataset_eval, idx_test.tolist()), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)
print("DataLoaders preparados.")

## 8. Visualización de ejemplos del entrenamiento

Estos ejemplos se obtendrán únicamente de las imágenes reales de train.
La visualización permite comprobar la lectura y el mapeo de clases; no
constituye una selección de aciertos o errores del modelo.

In [ ]:
def mostrar_imagenes(indices_muestra, dataset, titulos=None):
    if not len(indices_muestra):
        print("No hay imágenes para mostrar.")
        return
    columnas = min(4, len(indices_muestra))
    filas = math.ceil(len(indices_muestra) / columnas)
    fig, axes = plt.subplots(filas, columnas, figsize=(4 * columnas, 3.6 * filas), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for pos, indice in enumerate(indices_muestra):
        archivo, etiqueta = dataset.samples[int(indice)]
        with Image.open(archivo) as imagen:
            axes.flat[pos].imshow(imagen.convert("RGB"))
        axes.flat[pos].set_title(titulos[pos] if titulos else CLASES[etiqueta])
    fig.tight_layout()
    plt.show()

mostrar_imagenes(idx_train[:8], dataset_eval)

## 9. ResNet-18 y transfer learning

La red reutiliza características aprendidas en ImageNet. Se reemplaza la
capa final por cuatro salidas y se entrena inicialmente solo esa capa.
Opcionalmente se ajusta después el último bloque convolucional (`layer4`)
con una tasa de aprendizaje menor. El resto conserva sus pesos.

La red produce **logits**: la pérdida de entropía cruzada los recibe sin
softmax. Las probabilidades se calculan solo para mostrar la predicción.
Una puntuación softmax alta no garantiza que la imagen pertenezca a una de
las clases ni que la confianza esté calibrada.

In [ ]:
def crear_modelo(preentrenado=True):
    red = models.resnet18(weights=PESOS if preentrenado else None)
    for parametro in red.parameters():
        parametro.requires_grad = False
    red.fc = nn.Linear(red.fc.in_features, len(CLASES))
    return red.to(DEVICE)

def activar_fine_tuning(red):
    for parametro in red.layer4.parameters():
        parametro.requires_grad = True

def modo_entrenamiento(red):
    red.train()
    # Las estadísticas BatchNorm de capas congeladas tampoco deben cambiar.
    for modulo in red.modules():
        if isinstance(modulo, nn.modules.batchnorm._BatchNorm):
            if not any(p.requires_grad for p in modulo.parameters()):
                modulo.eval()

## 10. Pérdida, optimizador, validación y early stopping

Se utiliza `CrossEntropyLoss` y AdamW sobre los parámetros entrenables.
La validación se hace con `eval()` y sin gradientes. El early stopping
vigila la pérdida de validación y restaura la mejor configuración; no
utiliza el conjunto de test. Se registra pérdida y accuracy por época,
sin asignar ningún valor hasta que exista un entrenamiento real.

In [ ]:
criterio = nn.CrossEntropyLoss()

@torch.inference_mode()
def validar(red, loader):
    red.eval()
    perdida, aciertos, total = 0.0, 0, 0
    for imagenes, objetivos in loader:
        imagenes, objetivos = imagenes.to(DEVICE), objetivos.to(DEVICE)
        logits = red(imagenes)
        perdida += criterio(logits, objetivos).item() * len(objetivos)
        aciertos += (logits.argmax(1) == objetivos).sum().item()
        total += len(objetivos)
    if total == 0:
        raise ValueError("El DataLoader está vacío.")
    return perdida / total, aciertos / total

def entrenar_etapa(red, epochs, lr, etapa, conservar_inicial=False):
    optimizador = optim.AdamW((p for p in red.parameters() if p.requires_grad),
                             lr=lr, weight_decay=WEIGHT_DECAY)
    mejor_perdida = validar(red, val_loader)[0] if conservar_inicial else float("inf")
    mejores_pesos = copy.deepcopy(red.state_dict())
    sin_mejora, registros = 0, []
    for epoch in range(1, epochs + 1):
        inicio = time.perf_counter()
        modo_entrenamiento(red)
        perdida, aciertos, total = 0.0, 0, 0
        for imagenes, objetivos in train_loader:
            imagenes, objetivos = imagenes.to(DEVICE), objetivos.to(DEVICE)
            optimizador.zero_grad(set_to_none=True)
            logits = red(imagenes)
            loss = criterio(logits, objetivos)
            if not torch.isfinite(loss):
                raise RuntimeError("Pérdida no finita; revisar imágenes y configuración.")
            loss.backward()
            optimizador.step()
            perdida += loss.item() * len(objetivos)
            aciertos += (logits.argmax(1) == objetivos).sum().item()
            total += len(objetivos)
        val_loss, val_acc = validar(red, val_loader)
        fila = {"etapa": etapa, "epoch": epoch, "train_loss": perdida / total,
                "train_acc": aciertos / total, "val_loss": val_loss, "val_acc": val_acc,
                "segundos": time.perf_counter() - inicio}
        registros.append(fila)
        print(f"{etapa} {epoch}/{epochs} | train loss {fila['train_loss']:.4f} | "
              f"val loss {val_loss:.4f} | val acc {val_acc:.4f}", flush=True)
        if val_loss < mejor_perdida - MIN_DELTA:
            mejor_perdida = val_loss
            mejores_pesos = copy.deepcopy(red.state_dict())
            sin_mejora = 0
        else:
            sin_mejora += 1
        if sin_mejora >= PACIENCIA:
            print("Early stopping: se recuperan los mejores pesos de validación.")
            break
    red.load_state_dict(mejores_pesos)
    red.eval()
    return registros

## 11. Ejecución futura del entrenamiento

Esta celda requiere `ENTRENAR = True` y un dataset válido. La fase opcional
de fine-tuning conserva el modelo anterior si no mejora la validación.
No se ejecutará en la preparación actual. Si se interrumpe el entrenamiento,
no se marca el modelo como entrenado ni se generan métricas de test.

In [ ]:
if ENTRENAR:
    if not DATASET_OK:
        raise DatasetPendiente("Falta el dataset validado.")
    MODELO_ENTRENADO = False
    modelo = crear_modelo(preentrenado=True)
    historial = entrenar_etapa(modelo, EPOCHS_CABEZA, LR_CABEZA, "cabeza")
    if HACER_FINE_TUNING:
        activar_fine_tuning(modelo)
        historial += entrenar_etapa(modelo, EPOCHS_FINE_TUNING, LR_FINE_TUNING,
                                    "fine_tuning", conservar_inicial=True)
    MODELO_ENTRENADO = bool(historial)
    print("Entrenamiento finalizado. Mejores pesos recuperados.")
else:
    print("Entrenamiento desactivado. Activa ENTRENAR cuando el grupo entregue el dataset.")

## 12. Curvas de aprendizaje

Se representarán únicamente los registros generados por el entrenamiento.
La comparación entre train y validación permitirá estudiar el aprendizaje
y el posible sobreajuste. La interpretación de resultados queda pendiente.

In [ ]:
if MODELO_ENTRENADO and historial:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    pasos = range(1, len(historial) + 1)
    for ax, campo, titulo in zip(axes, ["loss", "acc"], ["Pérdida", "Accuracy"]):
        for conjunto in ["train", "val"]:
            ax.plot(pasos, [r[f"{conjunto}_{campo}"] for r in historial], label=conjunto)
        ax.set(xlabel="Época acumulada", ylabel=titulo, title=titulo)
        ax.legend()
        ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()
else:
    print("No hay un historial de entrenamiento real que representar.")

## 13. Guardado y carga del modelo

Se guarda un `state_dict`, las clases y su orden, las particiones, la
configuración y el historial real. Cada ejecución usa un nombre distinto
para evitar sobrescribir experimentos. La carga emplea `weights_only=True`
y verifica la correspondencia con las particiones actuales antes de permitir
la evaluación. Solo deben cargarse checkpoints propios y conocidos.

In [ ]:
def guardar_csv(ruta, filas):
    if filas:
        with Path(ruta).open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(filas[0]))
            writer.writeheader()
            writer.writerows(filas)

RUN_DIR = None
if MODELO_ENTRENADO and historial:
    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    RUN_DIR = RESULTADOS_PATH / run_id
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    checkpoint_path = MODELOS_PATH / f"resnet18_{run_id}.pt"
    checkpoint = {
        "state_dict": {k: v.detach().cpu() for k, v in modelo.state_dict().items()},
        "clases": CLASES, "class_to_idx": dataset_eval.class_to_idx,
        "entrenado": True, "historial": historial, "particiones": particiones,
        "configuracion": {"seed": SEED, "batch_size": BATCH_SIZE,
            "fracciones": [FRACCION_TRAIN, FRACCION_VAL, FRACCION_TEST],
            "lr_cabeza": LR_CABEZA, "lr_fine_tuning": LR_FINE_TUNING,
            "fine_tuning": HACER_FINE_TUNING, "umbral": UMBRAL_CONFIANZA,
            "pesos_iniciales": "IMAGENET1K_V1"},
        "versiones": {"torch": str(torch.__version__), "torchvision": str(torchvision.__version__)},
    }
    torch.save(checkpoint, checkpoint_path)
    guardar_csv(RUN_DIR / "historial.csv", historial)
    guardar_csv(RUN_DIR / "particiones.csv", particiones)
    print("Modelo guardado en:", checkpoint_path)

if CHECKPOINT_A_CARGAR is not None:
    if ENTRENAR:
        raise ValueError("Selecciona entrenamiento nuevo o carga de checkpoint, no ambos.")
    checkpoint = torch.load(Path(CHECKPOINT_A_CARGAR), map_location="cpu", weights_only=True)
    if checkpoint.get("clases") != CLASES or checkpoint.get("entrenado") is not True:
        raise ValueError("Checkpoint sin entrenamiento acreditado o con clases diferentes.")
    if checkpoint.get("particiones") != particiones:
        raise ValueError("Las particiones actuales no coinciden con el checkpoint. No evaluar sobre otro test.")
    modelo = crear_modelo(preentrenado=False)
    modelo.load_state_dict(checkpoint["state_dict"])
    modelo.eval()
    MODELO_ENTRENADO = True
    print("Modelo entrenado cargado correctamente.")

## 14. Evaluación sobre test

Activa `EVALUAR_TEST` una vez fijados los hiperparámetros mediante validación.
La celda exige un modelo realmente entrenado. Se calcula accuracy y
precision, recall y F1 **macro**, dando el mismo peso a cada clase; el informe
por clase muestra también su soporte. Los denominadores sin predicciones
válidas se tratan con `zero_division=0`. La matriz usa filas reales y
columnas predichas. No se interpreta ni se rellena ningún resultado antes
del experimento.

In [ ]:
@torch.inference_mode()
def predecir_loader(red, loader):
    red.eval()
    reales, predichas, confianzas = [], [], []
    for imagenes, objetivos in loader:
        probabilidades = red(imagenes.to(DEVICE)).softmax(dim=1)
        confianza, prediccion = probabilidades.max(dim=1)
        reales.extend(objetivos.tolist())
        predichas.extend(prediccion.cpu().tolist())
        confianzas.extend(confianza.cpu().tolist())
    return np.array(reales), np.array(predichas), np.array(confianzas)

resultados_test = None
if EVALUAR_TEST:
    if not MODELO_ENTRENADO or modelo is None:
        raise RuntimeError("Es necesario entrenar o cargar un modelo entrenado antes de evaluar.")
    y_true, y_pred, confianza_test = predecir_loader(modelo, test_loader)
    metricas = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }
    print(json.dumps(metricas, indent=2, ensure_ascii=False))
    informe = classification_report(y_true, y_pred, labels=list(range(4)),
                                    target_names=CLASES, zero_division=0)
    print(informe)
    matriz = confusion_matrix(y_true, y_pred, labels=list(range(4)))
    ConfusionMatrixDisplay(matriz, display_labels=CLASES).plot(cmap="Blues", values_format="d")
    plt.title("Matriz de confusión · test")
    plt.tight_layout()
    if RUN_DIR is None:
        RUN_DIR = RESULTADOS_PATH / ("evaluacion_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
        RUN_DIR.mkdir(parents=True, exist_ok=False)
    plt.savefig(RUN_DIR / "matriz_confusion.png", dpi=160, bbox_inches="tight")
    plt.show()
    (RUN_DIR / "metricas_test.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
    (RUN_DIR / "informe_clasificacion.txt").write_text(informe, encoding="utf-8")
    resultados_test = (y_true, y_pred, confianza_test)
    filas = [{"archivo": Path(dataset_eval.samples[int(idx_test[i])][0]).relative_to(DATASET_PATH).as_posix(),
              "real": CLASES[int(y_true[i])], "predicha": CLASES[int(y_pred[i])],
              "confianza": float(confianza_test[i])} for i in range(len(y_true))]
    guardar_csv(RUN_DIR / "predicciones_test.csv", filas)
else:
    print("Test pendiente. No se calculan métricas.")

## 15. Aciertos y errores reales

Tras la evaluación, se mostrarán los primeros ejemplos de cada grupo en el
orden de test. La selección no depende de buscar imágenes especialmente
favorables. Si un grupo está vacío, se indica expresamente. El análisis de
estos ejemplos corresponde a la fase posterior de resultados del grupo.

In [ ]:
if resultados_test is not None:
    y_true, y_pred, confianza_test = resultados_test
    for nombre, mascara in [("Aciertos", y_true == y_pred), ("Errores", y_true != y_pred)]:
        posiciones = np.flatnonzero(mascara)[:8]
        print(nombre)
        titulos = [f"Real: {CLASES[y_true[i]]}\nPred.: {CLASES[y_pred[i]]} ({confianza_test[i]:.1%})"
                   for i in posiciones]
        mostrar_imagenes(idx_test[posiciones], dataset_eval, titulos)
else:
    print("Aciertos y errores pendientes de una evaluación real.")

## 16. Predicción de una fotografía nueva

La fotografía debe contener un residuo principal. Se convierte a RGB y se
aplica exactamente el preprocesamiento de evaluación. La función devuelve
clase, confianza y distribución de probabilidades. `IMAGEN_A_PREDECIR` debe
ser una ruta relativa dentro de `pruebas/`. No se selecciona una imagen de
ejemplo mientras el modelo no esté entrenado.

In [ ]:
@torch.inference_mode()
def predecir_imagen(ruta, red):
    if not MODELO_ENTRENADO or red is None:
        raise RuntimeError("Primero entrena o carga un modelo entrenado.")
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"No existe la imagen: {ruta}")
    if ruta.suffix.lower() not in IMG_EXTENSIONS:
        raise ValueError("Extensión de imagen no admitida.")
    with Image.open(ruta) as imagen:
        tensor = transform_eval(imagen.convert("RGB")).unsqueeze(0).to(DEVICE)
    red.eval()
    probabilidades = red(tensor).softmax(dim=1)[0].cpu()
    indice = int(probabilidades.argmax())
    return {"clase": CLASES[indice], "confianza": float(probabilidades[indice]),
            "probabilidades": {clase: float(probabilidades[i]) for i, clase in enumerate(CLASES)}}

resultado_prediccion = None
if IMAGEN_A_PREDECIR is not None:
    resultado_prediccion = predecir_imagen(IMAGEN_A_PREDECIR, modelo)
    print("Predicción:", resultado_prediccion["clase"])
    print(f"Confianza: {resultado_prediccion['confianza']:.2%}")
else:
    print("No se ha seleccionado una fotografía nueva.")

## 17. Simulación de la decisión robótica

| Clase | Destino simulado |
|---|---|
| carton | Contenedor 1 |
| metal | Contenedor 2 |
| plastico | Contenedor 3 |
| vidrio | Contenedor 4 |

Si la confianza es inferior al umbral, la imagen se deriva a revisión manual.
La igualdad con el umbral permite la clasificación automática. Se validan
clase, confianza y umbral. Esta función solo genera una decisión: no envía
órdenes a ningún robot, PLC o dispositivo físico. El umbral inicial es
configurable y deberá justificarse con validación antes de su uso práctico.

In [ ]:
CONTENEDORES = {"carton": 1, "metal": 2, "plastico": 3, "vidrio": 4}

def decidir_accion(clase_predicha, confianza, umbral_minimo=UMBRAL_CONFIANZA):
    if clase_predicha not in CONTENEDORES:
        raise ValueError(f"Clase no admitida: {clase_predicha}")
    confianza, umbral_minimo = float(confianza), float(umbral_minimo)
    if not math.isfinite(confianza) or not 0 <= confianza <= 1:
        raise ValueError("La confianza debe ser un número finito entre 0 y 1.")
    if not math.isfinite(umbral_minimo) or not 0 <= umbral_minimo <= 1:
        raise ValueError("El umbral debe ser un número finito entre 0 y 1.")
    if confianza < umbral_minimo:
        return "revisión manual"
    return f"activar desviador hacia contenedor {CONTENEDORES[clase_predicha]}"

if resultado_prediccion is not None:
    accion = decidir_accion(resultado_prediccion["clase"], resultado_prediccion["confianza"])
    print("Predicción:", resultado_prediccion["clase"])
    print(f"Confianza: {resultado_prediccion['confianza']:.2%}")
    print("Acción:", accion)
else:
    print("Simulación preparada; pendiente de una predicción real.")

## 18. Continuación cuando se entregue el dataset

1. Incorporar las imágenes que entreguen los compañeros en las cuatro
   carpetas previstas, respetando su selección y etiquetado.
2. Reiniciar el kernel y ejecutar desde el principio. Revisar la comprobación
   de estructura y la viabilidad de la división. Si existen fotos del mismo
   objeto o sesión, acordar con el grupo una partición por grupos antes de entrenar.
3. Activar `ENTRENAR`, mantener `EVALUAR_TEST = False` y ejecutar el entrenamiento.
   Revisar la validación para fijar hiperparámetros y decidir sobre fine-tuning.
4. Guardar el modelo definitivo. Evaluar test cuando ya se haya cerrado la
   selección del modelo; no ajustar hiperparámetros a partir de test.
5. Probar fotografías nuevas y la decisión simulada. Entregar al grupo los
   archivos de resultados reales para su análisis y memoria.

**Pendiente:** validación funcional con el dataset definitivo, entrenamiento,
evaluación e interpretación de resultados. Los archivos de imágenes no se
modifican desde este notebook. No hay conclusiones experimentales todavía.

### Referencias técnicas de las APIs utilizadas

- [ResNet-18 y pesos preentrenados · Torchvision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
- [Transfer learning · PyTorch](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [División estratificada · scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [Métricas de clasificación · scikit-learn](https://scikit-learn.org/stable/modules/model_evaluation.html)